# NS10 Tutorial B - The Fitness Model

**Lecture:** NS10 - Preferential Attachment Extensions

**Reading:** Bianconi and Barabási (2001)

## Learning goals
- Use the slide rule
  $$
  P(i \leftrightarrow j) = \frac{\eta_j k_j}{\sum_l \eta_l k_l}.
  $$
- Understand why **quality differences** can overturn pure age dominance.
- Reproduce the **fit-get-rich** phenomenon computationally.
- Compare moderate and highly unequal fitness distributions.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def fitness_model(n, m, fitness_values, seed=RANDOM_SEED, tracked_nodes=None):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    history = []
    tracked_nodes = tracked_nodes or []

    for new_node in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        degrees = np.array([G.degree(node) for node in existing], dtype=float)
        fitness = np.array([fitness_values[node] for node in existing], dtype=float)
        weights = degrees * fitness
        probs = weights / weights.sum()
        targets = rng.choice(existing, size=m, replace=False, p=probs)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, int(target))

        if tracked_nodes:
            row = {'time t': new_node + 1}
            for node in tracked_nodes:
                if node in G:
                    row[f'degree_{node}'] = G.degree(node)
            history.append(row)

    return G, pd.DataFrame(history)


def make_fitness_vector(n, mode='uniform', seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    if mode == 'uniform':
        return rng.uniform(0.7, 1.3, size=n)
    if mode == 'skewed':
        return np.clip(rng.lognormal(mean=0.0, sigma=0.6, size=n), 0.4, 3.0)
    raise ValueError('mode must be uniform or skewed')


def node_table(G, fitness_values, top=10):
    rows = []
    for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:top]:
        rows.append({
            'node': node,
            'birth time': node + 1,
            'fitness': fitness_values[node],
            'degree': degree,
        })
    return pd.DataFrame(rows)

---
## 1. A late but high-fitness node can overtake an earlier one

The slide claim is that fitness changes the competition itself. We make that concrete by tracking two nodes:
- node 5: early arrival, but low fitness,
- node 80: later arrival, but very high fitness.


In [ ]:
n_demo = 800
m_demo = 2
fitness_demo = make_fitness_vector(n_demo, mode='uniform', seed=RANDOM_SEED)
fitness_demo[5] = 0.7
fitness_demo[80] = 1.9

demo_graph, demo_history = fitness_model(
    n_demo,
    m_demo,
    fitness_demo,
    seed=RANDOM_SEED,
    tracked_nodes=[5, 80],
)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(demo_history['time t'], demo_history['degree_5'], color=CATEGORY_PALETTE['blue'], linewidth=2, label='node 5 (fitness 0.7)')
ax.plot(demo_history['time t'], demo_history['degree_80'], color=CATEGORY_PALETTE['red'], linewidth=2, label='node 80 (fitness 1.9)')
style_axis(
    ax,
    title='Fit-get-rich: a later but fitter node can overtake an earlier one',
    xlabel='Network time t',
    ylabel='Degree',
    legend=True,
)
plt.show()

overtake_time = demo_history.loc[demo_history['degree_80'] > demo_history['degree_5'], 'time t'].iloc[0]
print(f'Overtake time: t = {int(overtake_time)}')
print(f'Final degree of node 5  : {demo_graph.degree(5)}')
print(f'Final degree of node 80 : {demo_graph.degree(80)}')


---
## 2. The degree distribution is no longer governed by one universal exponent

We compare two fitness regimes:
- **uniform** fitness: moderate quality differences,
- **skewed** fitness: stronger inequality.

The point is not to estimate a single perfect exponent. The point is to see that the tail depends on the fitness distribution itself.


In [ ]:
n_large = 1500
m_large = 2

fitness_uniform = make_fitness_vector(n_large, mode='uniform', seed=RANDOM_SEED)
fitness_skewed = make_fitness_vector(n_large, mode='skewed', seed=RANDOM_SEED)

graph_uniform, _ = fitness_model(n_large, m_large, fitness_uniform, seed=RANDOM_SEED)
graph_skewed, _ = fitness_model(n_large, m_large, fitness_skewed, seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
plot_ccdf([degree for _, degree in graph_uniform.degree()], ax=ax, label='Uniform fitness', color=CATEGORY_PALETTE['blue'], markersize=3)
plot_ccdf([degree for _, degree in graph_skewed.degree()], ax=ax, label='Skewed fitness', color=CATEGORY_PALETTE['orange'], markersize=3)
ax.set_title('Fitness distribution changes the tail shape')
ax.legend(frameon=False)
plt.show()


In [ ]:
rows = []
for label, graph, fitness_values in [
    ('Uniform fitness', graph_uniform, fitness_uniform),
    ('Skewed fitness', graph_skewed, fitness_skewed),
]:
    winner = max(graph.degree, key=lambda item: item[1])[0]
    degrees = np.array([degree for _, degree in graph.degree()])
    fitness_arr = np.array([fitness_values[node] for node in graph.nodes()])
    row = model_summary_row(graph, label)
    row['winner birth time'] = winner + 1
    row['winner fitness'] = fitness_values[winner]
    row['corr(fitness, degree)'] = np.corrcoef(fitness_arr, degrees)[0, 1]
    rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.round(3).to_string(index=False))


---
## 3. Quality-sensitive competition in the skewed regime

The skewed case is where the fit-get-rich effect is most visible. We inspect the joint distribution of fitness and final degree.


In [ ]:
skew_degree = np.array([graph_skewed.degree(node) for node in graph_skewed.nodes()])
skew_birth = np.array([node + 1 for node in graph_skewed.nodes()])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(fitness_skewed, skew_degree, color=CATEGORY_PALETTE['blue'], alpha=0.45, s=20)
style_axis(
    axes[0],
    title='Final degree versus fitness',
    xlabel='Fitness eta',
    ylabel='Final degree',
)

axes[1].scatter(skew_birth, skew_degree, c=fitness_skewed, cmap='Blues', alpha=0.6, s=20)
style_axis(
    axes[1],
    title='Birth time versus final degree (colour = fitness)',
    xlabel='Birth time',
    ylabel='Final degree',
)

plt.show()

print('Top nodes in the skewed-fitness run:')
print(node_table(graph_skewed, fitness_skewed, top=12).round(3).to_string(index=False))


## Takeaway

- The fitness model combines **cumulative advantage** and **intrinsic advantage**.
- A late node can become a hub if its fitness is high enough.
- There is no single universal exponent independent of the fitness distribution.
- This is the right extension when empirical systems reward quality, not just age and accumulated visibility.
